In [ ]:
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cv2

In [ ]:
from collections import Counter
from pathlib import Path

# function to determine the number of instances of each class
def sort_classes(folder_path):
    counts = Counter()
    for txt_file in Path(folder_path).glob("*.txt"):
        with open(txt_file, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    class_id = int(line.split()[0])
                    counts[class_id] += 1
                except (ValueError, IndexError):
                    pass  # skip malformed rows

    return dict(sorted(counts.items()))

In [ ]:
import pandas as pd
from pathlib import Path

train_path = Path(r'C:\Users\bdsc_\Documents\Computer-Vision-Pipeline\data\weld_data\train\images')
val_path = Path(r'C:\Users\bdsc_\Documents\Computer-Vision-Pipeline\data\weld_data\valid\images')
test_path = Path(r'C:\Users\bdsc_\Documents\Computer-Vision-Pipeline\data\weld_data\test\images')


def create_dataframe(path):
    paths = [
        p.parts[-2:]
        for p in path.rglob('*.*')
        if p.is_file()
    ]

    df = pd.DataFrame(
        paths,
        columns=['Class', 'Images']
    )

    df = df.sort_values(
        'Class',
        ascending=True
    )

    df.reset_index(
        drop=True,
        inplace=True
    )

    return df


train_df = create_dataframe(train_path)
val_df = create_dataframe(val_path)
test_df = create_dataframe(test_path)


print(train_df)
print(val_df)
print(test_df)

In [ ]:
{
  "names": {
    "0": "Porosity",
    "1": "Inclusion",
    "2": "Undercut",
    "3": "Burn-through",
    "4": "Crack",
    "5": "Overlap",
    "6": "Reference 1",
    "7": "Reference 2",
    "8": "Reference 3",
    "9": "Hidden Porosity",
    "10": "Shrinkage",
    "11": "Lack of Fusion",
    "12": "Incomplete Root Penetration"
  }
}

In [ ]:
print(sort_classes(r'C:\Users\Administrator\Downloads\Computer-Vision-Pipeline\data\weld_data\train\labels'))


In [ ]:
print(sort_classes(r'C:\Users\Administrator\Downloads\Computer-Vision-Pipeline\data\weld_data\test\labels'))
print(sort_classes(r'C:\Users\Administrator\Downloads\Computer-Vision-Pipeline\data\weld_data\valid\labels'))



Checking for images with missing label files

In [ ]:
from pathlib import Path


def get_images_without_labels(image_folder, label_folder):
    image_folder = Path(image_folder)
    label_folder = Path(label_folder)

    images_without_labels = []

    for img_path in image_folder.glob("*.*"):
        label_path = label_folder / f"{img_path.stem}.txt"

        if not label_path.exists():
            images_without_labels.append(img_path.name)

    return images_without_labels

In [ ]:
images = get_images_without_labels(
    r'C:\Users\bdsc_\Documents\Computer-Vision-Pipeline\data\weld_data\test\images',
    r'C:\Users\bdsc_\Documents\Computer-Vision-Pipeline\data\weld_data\test\labels'
)

print(len(images))
print(images[:10])

Checking for images with no actual defects 

In [ ]:
from pathlib import Path

def count_empty_labels(label_folder):
    count = 0

    for txt_file in Path(label_folder).glob("*.txt"):
        if txt_file.stat().st_size == 0:
            count += 1

    return count

In [ ]:
print(count_empty_labels(r'C:\Users\bdsc_\Documents\Computer-Vision-Pipeline\data\weld_data\train\labels'))
print(count_empty_labels(r'C:\Users\bdsc_\Documents\Computer-Vision-Pipeline\data\weld_data\val\labels'))
print(count_empty_labels(r'C:\Users\bdsc_\Documents\Computer-Vision-Pipeline\data\weld_data\test\labels'))


In [ ]:
from pathlib import Path
from collections import Counter


def count_images_per_class(folder_path):
    counts = Counter()

    for txt_file in Path(folder_path).glob("*.txt"):
        image_classes = set()

        with open(txt_file, "r") as f:
            for line in f:
                line = line.strip()

                if not line:
                    continue

                try:
                    class_id = int(line.split()[0])
                    image_classes.add(class_id)

                except (ValueError, IndexError):
                    continue

        # count the image once for every class present
        for class_id in image_classes:
            counts[class_id] += 1

    return dict(sorted(counts.items()))

In [ ]:
count_images_per_class(r'C:\Users\Administrator\Downloads\Computer-Vision-Pipeline\data\weld_data\train\labels')

In [ ]:
import json
import pandas as pd
import numpy as np
from collections import defaultdict

# Load COCO annotations
with open(r"C:\Users\Administrator\Downloads\Computer-Vision-Pipeline\data\weld_data\instances_train.json", "r") as f:
    coco = json.load(f)

# Category ID -> Category Name
cat_map = {c["id"]: c["name"] for c in coco["categories"]}

# Collect classes present in each image
image_classes = defaultdict(set)

for ann in coco["annotations"]:
    image_id = ann["image_id"]
    class_name = cat_map[ann["category_id"]]
    image_classes[image_id].add(class_name)

# Create binary image-class matrix
all_classes = sorted(cat_map.values())

rows = []
for image_id, classes in image_classes.items():
    row = {cls: int(cls in classes) for cls in all_classes}
    row["image_id"] = image_id
    rows.append(row)

df = pd.DataFrame(rows).set_index("image_id")

print(df.head())

Pearson Coefficient Correlation

In [ ]:
corr_matrix = df.corr()

print(corr_matrix)
sns.heatmap(corr_matrix)

In [ ]:
Jaccard Probability Correlation

In [ ]:
from sklearn.metrics import jaccard_score

classes = df.columns

jaccard = pd.DataFrame(
    index=classes,
    columns=classes,
    dtype=float
)

for c1 in classes:
    for c2 in classes:
        jaccard.loc[c1, c2] = jaccard_score(
            df[c1],
            df[c2]
        )

print(jaccard.round(3))
sns.heatmap(jaccard)

Conditional Probability Correlation

In [ ]:
classes = df.columns

cond_prob = pd.DataFrame(
    index=classes,
    columns=classes,
    dtype=float
)

for a in classes:
    for b in classes:

        count_b = df[b].sum()

        if count_b == 0:
            cond_prob.loc[a, b] = np.nan
        else:
            count_ab = ((df[a] == 1) & (df[b] == 1)).sum()
            cond_prob.loc[a, b] = count_ab / count_b

print(cond_prob.round(3))
sns.heatmap(cond_prob)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


def plot_class_distribution(class_counts, class_names=None, title="Class Distribution"):
    
    classes = list(class_counts.keys())
    counts = list(class_counts.values())

    if class_names:
        labels = [class_names[c] for c in classes]
    else:
        labels = classes

    plt.figure(figsize=(12, 6))

    sns.barplot(
        x=labels,
        y=counts
    )

    plt.xticks(
        rotation=45,
        ha="right"
    )

    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.title(title)

    # add values above bars
    for i, count in enumerate(counts):
        plt.text(
            i,
            count,
            str(count),
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    plt.show()

In [ ]:
class_names = {
    0: "Porosity",
    1: "Inclusion",
    2: "Undercut",
    3: "Burn-through",
    6: "Reference 1",
    7: "Reference 2",
    8: "Reference 3",
    10: "Shrinkage",
    11: "Lack of Fusion",
    12: "Root Penetration"
}

instances = {
    0: 24426,
    1: 5895,
    2: 395,
    3: 1,
    6: 2586,
    7: 2255,
    8: 5,
    10: 97,
    11: 191,
    12: 118
}

plot_class_distribution(
    instances,
    class_names,
    "Welding Defect Instances"
)

In [ ]:
import pandas as pd


def compare_distribution(image_counts, instance_counts, class_names):

    df = pd.DataFrame({
        "Images": image_counts,
        "Instances": instance_counts
    })

    df["Class"] = [
        class_names[i]
        for i in df.index
    ]

    df = df.reset_index(drop=True)

    df.plot(
        x="Class",
        kind="bar",
        figsize=(12,6)
    )

    plt.xticks(
        rotation=45,
        ha="right"
    )

    plt.ylabel("Count")
    plt.title("Images vs Instances per Class")

    plt.tight_layout()
    plt.show()

In [ ]:
class_count = count_images_per_class(r'C:\Users\Administrator\Downloads\Computer-Vision-Pipeline\data\weld_data\train\labels')
print(class_count)

In [ ]:
compare_distribution(class_count, instances, class_names)